Pada bagian ini Kami mengupload file kaggle.json yang berisi API key dari Kaggle untuk mengakses dataset secara langsung dari Kaggle.

In [ ]:
from google.colab import files
files.upload()

Kemudian dilakukan konfigurasi Kaggle API dengan membuat folder .kaggle, menyalin file API key, dan mengatur permission agar dapat digunakan dengan aman.

In [ ]:
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json
print("✅ Kaggle API siap!")

Dataset diunduh langsung dari Kaggle menggunakan API, yaitu dataset PlantVillage yang berisi gambar daun dengan berbagai jenis penyakit

In [ ]:
!kaggle datasets download -d abdallahalidev/plantvillage-dataset
print("✅ Download selesai!")

Setelah diunduh, dataset diekstrak ke dalam folder /content/dataset agar dapat digunakan dalam proses training.

In [ ]:
!unzip plantvillage-dataset.zip -d /content/dataset
print("✅ Extract selesai!")

Pada bagian ini, Kami membuat fungsi untuk mengeksplorasi struktur dataset menggunakan os.walk.
Tujuannya adalah untuk memahami bagaimana data disusun sebelum digunakan untuk training.

In [ ]:
import os

dataset_path = '/content/dataset'
for root, dirs, files in os.walk(dataset_path):
    level = root.replace(dataset_path, '').count(os.sep)
    indent = ' ' * 2 * level
    print(f"{indent}{os.path.basename(root)}/")
    if level < 2:
        for f in files[:3]:
            print(f"{indent}  {f}")

Lanjutan Yang Atas
Dari hasil ini, terlihat bahwa dataset memiliki beberapa folder utama seperti grayscale, color, dan segmented.

Setiap folder berisi subfolder yang merepresentasikan kelas, misalnya jenis tanaman dan penyakit seperti Tomato Early Blight atau Apple Healthy.

Pada project ini, Kami memilih menggunakan data dari folder color karena memiliki informasi visual yang lebih lengkap dibandingkan grayscale.


Di sini Kami menginstall library timm, yang digunakan untuk mengakses berbagai model pretrained dalam Computer Vision.

In [ ]:
!pip install timm -q

Kemudian Kami mengimport library yang dibutuhkan seperti PyTorch untuk membangun model, torchvision untuk pengolahan gambar, dan sklearn untuk evaluasi model.

In [ ]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
import timm
import numpy as np
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
from sklearn.metrics import classification_report
from tqdm import tqdm

Pada bagian ini Kami mendefinisikan parameter training seperti batch size, jumlah epoch, learning rate, dan ukuran gambar yang akan digunakan sebagai input model.
Sama disini kami menentukan device yang dipake yaitu pake gpu(Cuda) ini akan lebih cepat dibanding CPU

In [ ]:
path = "/content/dataset" # change path
BATCH_SIZE = 32
EPOCHS = 15
LR = 0.0001
IMG_SIZE = 224

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Pada bagian ini, saya menentukan path dataset yang digunakan, yaitu folder color dari PlantVillage. Kemudian saya menampilkan daftar kelas yang tersedia, di mana setiap folder merepresentasikan jenis tanaman dan penyakitnya.(Data_Root),Dari outputnya terlihat banyak kelas misal Raspeberry_helathy dll.

Selanjutnya dilakukan preprocessing dan data augmentation. Gambar di-resize agar seragam, lalu dilakukan augmentasi seperti flipping, rotation, dan color jitter untuk menambah variasi data dan meningkatkan kemampuan generalisasi model. (Ini pada bagian train transform)

Untuk data validation, hanya dilakukan resize tanpa augmentasi agar evaluasi model lebih objektif. Ini Pada val_transforms

Dataset dimuat menggunakan ImageFolder, yang secara otomatis membaca label berdasarkan nama folder (full_dataset)

setelah itu dataset dibagi menjadi training dan validation dengan rasio 80 : 20 untuk melatih modelnya

DataLoader digunakan untuk membagi data ke dalam batch agar proses training lebih efisien dan dapat dilakukan secara Berulang.

In [ ]:
import os
from torchvision import datasets
from torch.utils.data import DataLoader

# The path based on your specific 'color' subdirectory find
data_root = "/content/dataset/plantvillage dataset/color"
# Verify the contents—you should see folder names like 'Apple___Apple_scab', etc.
print("Actual classes found:", os.listdir(data_root))
#transormasi -> ini yang ada dimateri compvis
train_transforms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(20),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
])

val_transforms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
])

# Load the dataset
# Note: If this folder contains the classes directly (no 'train'/'val' split),
# you'll need to manually split the dataset.
full_dataset = datasets.ImageFolder(root=data_root, transform=train_transforms)

# Example: 80/20 Split
train_size = int(0.8 * len(full_dataset))
val_size = len(full_dataset) - train_size
train_dataset, val_dataset = torch.utils.data.random_split(full_dataset, [train_size, val_size])

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)

print(f"Total images: {len(full_dataset)}")

Di sini Kami cuma ngecek apakah folder dataset sudah terbaca dengan benar.

In [ ]:
import os

for root, dirs, files in os.walk("/content/dataset"):
    print(root)
    break

menampilkan isi folder dataset di Colab

In [ ]:
!ls /content/dataset

menampilkan isi folder “plantvillage dataset”

In [ ]:
!ls "/content/dataset/plantvillage dataset"

Pada bagian ini, dataset dimuat menggunakan ImageFolder, di mana label otomatis diambil dari nama folder kelas (full_dataset)

Dataset kemudian dibagi menjadi data training dan validation dengan rasio 80:20 untuk melatih dan mengevaluasi model.

Setelah split, saya mengganti transform pada data validation agar tidak menggunakan augmentasi, sehingga evaluasi model tetap objektif (Pada Val_dataset)

DataLoader digunakan untuk membagi data ke dalam batch agar proses training lebih efisien

In [ ]:
from torch.utils.data import random_split

DATA_DIR = "/content/dataset/plantvillage dataset/color"

full_dataset = datasets.ImageFolder(DATA_DIR, transform=train_transforms)

# 80-20 split
train_size = int(0.8 * len(full_dataset))
val_size = len(full_dataset) - train_size

train_dataset, val_dataset = random_split(full_dataset, [train_size, val_size])

# Apply validation transform separately
val_dataset.dataset.transform = val_transforms

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)

num_classes = len(full_dataset.classes)
print("Classes:", full_dataset.classes)

Pada bagian ini, Kami menggunakan model EfficientNet-B0 dari library timm dengan pretrained weights, sehingga model sudah memiliki pengetahuan awal dari dataset besar seperti ImageNet.

Layer terakhir pada model diganti agar sesuai dengan jumlah kelas pada dataset saya, sehingga model dapat melakukan klasifikasi sesuai dengan jumlah penyakit tanaman

Model dijalankan dalam model GPU

In [ ]:
model = timm.create_model("efficientnet_b0", pretrained=True)
model.classifier = nn.Linear(model.classifier.in_features, num_classes)
model = model.to(device)

Pada bagian ini, saya menggunakan CrossEntropyLoss sebagai fungsi loss untuk mengukur seberapa besar kesalahan prediksi model terhadap label sebenarnya

Saya menggunakan optimizer Adam untuk mengupdate bobot model selama training, dengan learning rate sebesar 0.0001 agar proses pembelajaran berjalan stabil

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=LR)

Fungsi ini digunakan untuk melatih model selama satu epoch, yaitu satu kali proses terhadap seluruh data training. (Def train_epoch)

Model diatur ke mode training agar layer seperti dropout dan batch normalization bekerja sesuai kebutuhan training. (Model.train) , Kemudian ada inisialisasi 3 variabel untuk nyimpen total loss sama dia hitung akurasi selama dia di train

Untuk for images,labels itu Data diambil per batch dari DataLoader untuk diproses secara bertahap.,Lalu ada images.to device itu kami pindahin ke GPU biar bisa lebih cepat.

Setelah itu ada optimizer itu untuk Gradient di-reset agar tidak menumpuk dari iterasi sebelumnya,yang outputs itu dia prediksi sesuai gambar,lalu ada loss itu untuk menghitung kesalahan prediksi modelnya,sama loss backward itu backpropogation utk hitung gradient dari loss dan yang terakhir optimizer step itu dia perbarui bobotnya

Selanjutnya itu ada predicted itu untuk prediksi kelas diambil dari nilai tertinggi,kemudian yang correct itu untuk menghitung jumlah prediksi yang benar

In [ ]:
def train_one_epoch():
    model.train()
    running_loss = 0
    correct = 0
    total = 0

    for images, labels in tqdm(train_loader):
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()

    accuracy = 100. * correct / total
    return running_loss/len(train_loader), accuracy

Fungsi validate itu digunakan untuk evaluasi performa,kemudian Model diubah ke mode evaluasi agar layer seperti dropout dan batch normalization tidak aktif seperti saat training,inisilasi variabel kayak sebelumnya untuk hitung loss dan akurasi,kemudian yang allpreds dikasih [], alllabels itu untuk nyimpen hasil prediksi dan label asli

Selanjutnya itu ada nograd itu pada validasi,gradient tidak dihitung,karena model tidak lagi belajar,habis itu ada for images ,labels itu dia diproses per batch pake dataloader

untuk yang outputs itu dia prediksi terhadap data validationnya,lalu ada loss lagi dia mengukur performa modelnya,Lalu yang predicted dia ambil kelas prediksi dari nilai yang tertinggi,lalu di correct dia itung jumlah prediksi yang benar,allpreds dan alllabels itu prediksi dan label disimpan

In [ ]:
def validate():
    model.eval()
    running_loss = 0
    correct = 0
    total = 0

    all_preds = []
    all_labels = []

    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)

            running_loss += loss.item()
            _, predicted = outputs.max(1)

            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()

            all_preds.extend(predicted.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    accuracy = 100. * correct / total
    return running_loss/len(val_loader), accuracy, all_preds, all_labels

Pada Bagian bestacc ini diinisialisasi untuk menyimpan akurasi terbaik,serta parameter early stopping yaitu patience ,trigger.
Setelah ini ada yang for epoch itu untuk melatih model dengan beberapa epoch,Selanjutnya pada proses epoch dilakukan training dan validation untuk lihat performa modelnya.

untuk yang jika vall acc lebih besar dari best acc itu jika akurasi val naik ,dia disimpan jadi model terbaik.lalu pada torch save itu model disimpan agar bisa digunakan kembali.sama yang terakhir jika tidak ada peningkatan pas beberapa epoch itu langsung lakukan percepatan stop,biar gak overfitting.

In [ ]:
best_acc = 0
patience = 3
trigger = 0

for epoch in range(EPOCHS):
    print(f"\nEpoch {epoch+1}/{EPOCHS}")

    train_loss, train_acc = train_one_epoch()
    val_loss, val_acc, preds, labels = validate()

    print(f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.2f}%")
    print(f"Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.2f}%")

    if val_acc > best_acc:
        best_acc = val_acc
        torch.save(model.state_dict(), "efficientnet_plant_best.pth")
        print("✅ Model Saved!")
        trigger = 0
    else:
        trigger += 1
        if trigger >= patience:
            print("⛔ Early Stopping Triggered")
            break

Dibagian ini Kami memunculkan laporan klasifikasi yaitu berupa precision,recall,f1-score,dan support.target names dilakukan agar namanya sama dengan label asli

In [ ]:
print("\nClassification Report:")
print(classification_report(labels, preds, target_names=full_dataset.classes))
print("Training Complete 🚀")

Lalu disini cek isi folder lagi dengan tampilin 5 pertama

In [ ]:
import os

# Lihat isi folder dulu
folder = "/content/dataset/plantvillage dataset/color/Apple___Apple_scab"
print(os.listdir(folder)[:5])

Disini digunakan untuk mengambil contoh data dan memastikan bahwa path file valid.Supaya bisa di pake pas selanjutnya

In [ ]:
folder = "/content/dataset/plantvillage dataset/color/Apple___Apple_scab"
filename = os.listdir(folder)[0]
image_path = os.path.join(folder, filename)

print("Image path:", image_path)

Pada bagian ini, kami memuat kembali model yang sudah dilatih sebelumnya menggunakan file weight yang telah disimpan. (Ini pada model,classifier,load).selanjutnya model diatur ke bentuk eval karena ini lagi prediksi bukan training.

kemudian pada transform Gambar di-resize dan diubah menjadi tensor agar sesuai dengan input model.”,Setelah dari ini imagenya di convert menjadi bentuk RGB.Selanjut pada input tensor itu Ditambahkan dimensi batch karena model membutuhkan input dalam bentuk batch.

Pada outputs,probs,confidence,predicted itu Model menghasilkan prediksi, kemudian diubah menjadi probabilitas menggunakan softmax. Kelas dengan nilai tertinggi dipilih sebagai hasil prediksi.,Kemudian yang predicted class itu index hasilnya dirubah menjadi nama kelas.

In [ ]:
import torch
import timm
from torchvision import transforms
from PIL import Image
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

# Use classes directly from dataset
class_names = full_dataset.classes

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Load model
model = timm.create_model("efficientnet_b0", pretrained=False)
model.classifier = nn.Linear(model.classifier.in_features, len(class_names))
model.load_state_dict(torch.load("efficientnet_plant_best.pth", map_location=device))
model.to(device)
model.eval()

# CHANGE THIS PATH
image_path = "/content/dataset/plantvillage dataset/color/Apple___Apple_scab/276ed34e-9987-4b38-b83b-8626504fc204___FREC_Scab 3050.JPG"
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
])

image = Image.open(image_path).convert("RGB")
input_tensor = transform(image).unsqueeze(0).to(device)

with torch.no_grad():
    outputs = model(input_tensor)
    probs = F.softmax(outputs, dim=1)
    confidence, predicted = torch.max(probs, 1)

predicted_class = class_names[predicted.item()]
confidence_score = confidence.item() * 100

plt.imshow(image)
plt.axis("off")
plt.title(f"{predicted_class} ({confidence_score:.2f}%)")
plt.show()

print("Prediction:", predicted_class)
print("Confidence:", f"{confidence_score:.2f}%")

Pada model eval itu diatur ke mode eval itu karena kita mau predict,kemudian pada img,label kami mengambil gambar secara acak dari data validation untuk menguji model pada data yang belum pernah dilatih.

In [ ]:
import random
import matplotlib.pyplot as plt
import torch.nn.functional as F

model.eval()

for i in range(5):
    img, label = val_dataset[random.randint(0, len(val_dataset)-1)]
    img_tensor = img.unsqueeze(0).to(device)

    with torch.no_grad():
        outputs = model(img_tensor)
        probs = F.softmax(outputs, dim=1)
        conf, pred = torch.max(probs, 1)

    plt.imshow(img.permute(1,2,0))
    plt.axis("off")
    plt.title(f"Pred: {class_names[pred.item()]}\nTrue: {class_names[label]}\nConf: {conf.item()*100:.2f}%")
    plt.show()

menyimpan list nama kelas (label) ke file .json

In [ ]:
import json

class_names = full_dataset.classes

with open("class_names.json", "w") as f:
    json.dump(class_names, f)

print("class_names.json saved successfully!")

Mengunduh file dari colab

In [ ]:
from google.colab import files
files.download("/content/class_names.json")
files.download("/content/efficientnet_plant_best.pth")

In [5]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [7]:
import json

path = r"/content/drive/MyDrive/Colab Notebooks/Computer Vision .ipynb"

with open(path, "r", encoding="utf-8") as f:
    nb = json.load(f)

if "widgets" in nb.get("metadata", {}):
    del nb["metadata"]["widgets"]

with open(path, "w", encoding="utf-8") as f:
    json.dump(nb, f)

print("fixed")

fixed
